# Scratch Bi-LSTM

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from collections import Counter
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A', 'B', 'C', 'D', 'E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 1. Data Loading & Cleaning

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head()

In [ ]:
def clean_text(t):
    if pd.isna(t):
        return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

## 2. Leak-Free Train/Val Split (UnionFind)

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i

train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))

train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l: i for i, l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)

y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero overlap in option-sets)")

## 3. Custom Vocabulary & Tokenizer

In [ ]:
def tokenize(text):
    """Simple whitespace tokenizer on cleaned text."""
    return clean_text(text).split()


word_counts = Counter()
for _, row in train_split.iterrows():
    word_counts.update(tokenize(row['prompt']))
    for l in LABELS:
        word_counts.update(tokenize(row[l]))


MAX_VOCAB = 15000
PAD_IDX = 0
UNK_IDX = 1

vocab = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in word_counts.most_common(MAX_VOCAB):
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

def encode(text, max_len=64):
    """Convert text to token indices, truncate to max_len."""
    tokens = tokenize(text)[:max_len]
    return [vocab.get(w, UNK_IDX) for w in tokens]

## 4. PyTorch Dataset for Multiple Choice

In [ ]:
MAX_PROMPT_LEN = 128
MAX_OPTION_LEN = 64

class MCQDataset(Dataset):
    def __init__(self, df, labels=None):
        self.prompts = []
        self.options = []  
        self.labels = labels

        for _, row in df.iterrows():
            self.prompts.append(encode(row['prompt'], MAX_PROMPT_LEN))
            opts = [encode(row[l], MAX_OPTION_LEN) for l in LABELS]
            self.options.append(opts)

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = torch.tensor(self.prompts[idx], dtype=torch.long)
        options = [torch.tensor(o, dtype=torch.long) for o in self.options[idx]]
        if self.labels is not None:
            return prompt, options, self.labels[idx]
        return prompt, options


def collate_fn(batch):
    """Custom collation: pad prompts and each option separately."""
    has_labels = len(batch[0]) == 3

    prompts = [b[0] for b in batch]
    all_options = [b[1] for b in batch]

    
    prompt_lens = torch.tensor([len(p) for p in prompts])
    prompts_padded = pad_sequence(prompts, batch_first=True, padding_value=PAD_IDX)

    
    options_padded = []
    option_lens = []
    for opt_idx in range(5):
        opts = [b[opt_idx] for b in all_options]
        lens = torch.tensor([max(len(o), 1) for o in opts])
    
        opts = [o if len(o) > 0 else torch.tensor([UNK_IDX]) for o in opts]
        padded = pad_sequence(opts, batch_first=True, padding_value=PAD_IDX)
        options_padded.append(padded)
        option_lens.append(lens)

    if has_labels:
        labels = torch.tensor([b[2] for b in batch], dtype=torch.long)
        return prompts_padded, prompt_lens, options_padded, option_lens, labels

    return prompts_padded, prompt_lens, options_padded, option_lens

## 4. PyTorch Dataset for Multiple Choice

In [ ]:
MAX_PROMPT_LEN = 128
MAX_OPTION_LEN = 64

class MCQDataset(Dataset):
    def __init__(self, df, labels=None):
        self.prompts = []
        self.options = []  
        self.labels = labels

        for _, row in df.iterrows():
            self.prompts.append(encode(row['prompt'], MAX_PROMPT_LEN))
            opts = [encode(row[l], MAX_OPTION_LEN) for l in LABELS]
            self.options.append(opts)

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = torch.tensor(self.prompts[idx], dtype=torch.long)
        options = [torch.tensor(o, dtype=torch.long) for o in self.options[idx]]
        if self.labels is not None:
            return prompt, options, self.labels[idx]
        return prompt, options


def collate_fn(batch):
    """Custom collation: pad prompts and each option separately."""
    has_labels = len(batch[0]) == 3

    prompts = [b[0] for b in batch]
    all_options = [b[1] for b in batch]

    
    prompt_lens = torch.tensor([len(p) for p in prompts])
    prompts_padded = pad_sequence(prompts, batch_first=True, padding_value=PAD_IDX)

    
    options_padded = []
    option_lens = []
    for opt_idx in range(5):
        opts = [b[opt_idx] for b in all_options]
        lens = torch.tensor([max(len(o), 1) for o in opts])
    
        opts = [o if len(o) > 0 else torch.tensor([UNK_IDX]) for o in opts]
        padded = pad_sequence(opts, batch_first=True, padding_value=PAD_IDX)
        options_padded.append(padded)
        option_lens.append(lens)

    if has_labels:
        labels = torch.tensor([b[2] for b in batch], dtype=torch.long)
        return prompts_padded, prompt_lens, options_padded, option_lens, labels

    return prompts_padded, prompt_lens, options_padded, option_lens

## 5. Bi-LSTM Model Architecture

In [ ]:
class AttentionPool(nn.Module):
  
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, lengths):
        max_len = lstm_out.size(1)
        mask = torch.arange(max_len, device=lstm_out.device).unsqueeze(0) < lengths.unsqueeze(1)

        scores = self.attn(lstm_out).squeeze(-1)        
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1)         
        pooled = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)  
        return pooled


class BiLSTMScratchModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attn_pool = AttentionPool(hidden_dim * 2)  
        self.dropout = nn.Dropout(dropout)

        repr_dim = hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(repr_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, input_ids, lengths):
        """Encode a batch of sequences using Embedding + Bi-LSTM + Attention."""
        embedded = self.dropout(self.embedding(input_ids))

    
        packed = pack_padded_sequence(
            embedded, lengths.cpu().clamp(min=1),
            batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)

        pooled = self.attn_pool(lstm_out, lengths)
        return pooled

    def forward(self, prompt_ids, prompt_lens, option_ids_list, option_lens_list):
        prompt_repr = self.encode(prompt_ids, prompt_lens)  

        logits = []
        for opt_ids, opt_lens in zip(option_ids_list, option_lens_list):
            opt_repr = self.encode(opt_ids, opt_lens)       
            combined = torch.cat([
                prompt_repr,
                opt_repr,
                prompt_repr * opt_repr    
            ], dim=-1)
            score = self.classifier(combined)           
            logits.append(score)

        return torch.cat(logits, dim=-1)  

print("Model architecture defined.")
print(BiLSTMScratchModel(len(vocab)))

## 6. Evaluation Metric: MAP@3

In [ ]:
def map3_from_probs(probs, true_idx):
    """Compute Mean Average Precision @ 3."""
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(true_idx)):
        for rank, pred in enumerate(top3[i]):
            if pred == true_idx[i]:
                score += 1.0 / (rank + 1)
                break
    return score / len(true_idx)

## 7. Training Loop (5-Fold Cross-Validation with WandB)

In [ ]:
GROUP_NAME = "scratch-bilstm"
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3

gkf = GroupKFold(n_splits=5)
oof_probs = np.zeros((len(train_split), 5))
val_probs_list = []
test_probs_list = []


test_dataset = MCQDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

for fold, (tr_i, va_i) in enumerate(gkf.split(train_split, y_tr, groups)):
    print(f"\n{'='*50}")
    print(f"  FOLD {fold}")
    print(f"{'='*50}")

    wandb.init(
        project="smart-mcq-solver",
        entity="23f2004192-dl-genai-project",
        group=GROUP_NAME,
        job_type="cv_fold",
        name=f"fold_{fold}",
        config={
            "model": "BiLSTM-scratch",
            "epochs": EPOCHS,
            "lr": LR,
            "embed_dim": 128,
            "hidden_dim": 128,
            "num_layers": 2,
            "batch_size": BATCH_SIZE,
            "vocab_size": len(vocab),
            "max_prompt_len": MAX_PROMPT_LEN,
            "max_option_len": MAX_OPTION_LEN,
        }
    )

    fold_train = train_split.iloc[tr_i].reset_index(drop=True)
    fold_val   = train_split.iloc[va_i].reset_index(drop=True)

    train_dataset = MCQDataset(fold_train, labels=y_tr[tr_i])
    val_dataset   = MCQDataset(fold_val,   labels=y_tr[va_i])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    
    model = BiLSTMScratchModel(len(vocab)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_map3 = 0
    best_state = None

    for epoch in range(EPOCHS):
       
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f"F{fold} E{epoch+1}", leave=False):
            prompt_ids, prompt_lens, opt_ids, opt_lens, labels = batch
            prompt_ids = prompt_ids.to(device)
            prompt_lens = prompt_lens.to(device)
            opt_ids = [o.to(device) for o in opt_ids]
            opt_lens = [l.to(device) for l in opt_lens]
            labels = labels.to(device)

            logits = model(prompt_ids, prompt_lens, opt_ids, opt_lens)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)

       
        model.eval()
        all_probs = []
        all_labels = []
        val_loss_sum = 0.0

        with torch.no_grad():
            for batch in val_loader:
                prompt_ids, prompt_lens, opt_ids, opt_lens, labels = batch
                prompt_ids = prompt_ids.to(device)
                prompt_lens = prompt_lens.to(device)
                opt_ids = [o.to(device) for o in opt_ids]
                opt_lens = [l.to(device) for l in opt_lens]
                labels = labels.to(device)

                logits = model(prompt_ids, prompt_lens, opt_ids, opt_lens)
                val_loss_sum += criterion(logits, labels).item()
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.cpu().numpy())

        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)

        val_map3 = map3_from_probs(all_probs, all_labels)
        val_preds = all_probs.argmax(axis=1)
        val_acc = accuracy_score(all_labels, val_preds)
        val_f1 = f1_score(all_labels, val_preds, average='macro')
        avg_val_loss = val_loss_sum / len(val_loader)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_map3": val_map3,
            "val_accuracy": val_acc,
            "val_f1": val_f1,
        })

        print(f"  Fold {fold} Epoch {epoch+1}: train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f} | MAP@3={val_map3:.4f} | acc={val_acc:.4f}")

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

  
    model.load_state_dict(best_state)
    wandb.finish()
    print(f"  Fold {fold} best MAP@3: {best_map3:.4f}")

   
    model.eval()
    with torch.no_grad():
        oof_loader = DataLoader(
            MCQDataset(fold_val, labels=y_tr[va_i]),
            batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )
        fold_oof = []
        for batch in oof_loader:
            p_ids, p_lens, o_ids, o_lens, _ = batch
            p_ids = p_ids.to(device); p_lens = p_lens.to(device)
            o_ids = [o.to(device) for o in o_ids]
            o_lens = [l.to(device) for l in o_lens]
            fold_oof.append(torch.softmax(model(p_ids, p_lens, o_ids, o_lens), dim=-1).cpu().numpy())
        oof_probs[va_i] = np.concatenate(fold_oof)

        # Holdout val
        val_ds_full = MCQDataset(val_split)
        val_ldr_full = DataLoader(val_ds_full, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
        fold_val_probs = []
        for batch in val_ldr_full:
            p_ids, p_lens, o_ids, o_lens = batch
            p_ids = p_ids.to(device); p_lens = p_lens.to(device)
            o_ids = [o.to(device) for o in o_ids]
            o_lens = [l.to(device) for l in o_lens]
            fold_val_probs.append(torch.softmax(model(p_ids, p_lens, o_ids, o_lens), dim=-1).cpu().numpy())
        val_probs_list.append(np.concatenate(fold_val_probs))

        fold_test_probs = []
        for batch in test_loader:
            p_ids, p_lens, o_ids, o_lens = batch
            p_ids = p_ids.to(device); p_lens = p_lens.to(device)
            o_ids = [o.to(device) for o in o_ids]
            o_lens = [l.to(device) for l in o_lens]
            fold_test_probs.append(torch.softmax(model(p_ids, p_lens, o_ids, o_lens), dim=-1).cpu().numpy())
        test_probs_list.append(np.concatenate(fold_test_probs))

print("\nAll folds complete")

## 8. Final Evaluation

In [ ]:

val_probs = np.mean(val_probs_list, axis=0)
test_probs = np.mean(test_probs_list, axis=0)
val_true = val_split['label'].values

final_map3 = map3_from_probs(val_probs, val_true)
final_preds = np.argmax(val_probs, axis=1)
final_acc = accuracy_score(val_true, final_preds)
final_f1 = f1_score(val_true, final_preds, average='macro')

print(f"Scratch Bi-LSTM Val MAP@3:    {final_map3:.4f}")
print(f"Scratch Bi-LSTM Val Accuracy: {final_acc:.4f}")
print(f"Scratch Bi-LSTM Val Macro F1: {final_f1:.4f}")


wandb.init(
    project="smart-mcq-solver",
    entity="23f2004192-dl-genai-project",
    group=GROUP_NAME,
    job_type="summary",
    name="final_summary",
    tags=["scratch-bilstm", "final"],
    config={"model": "BiLSTM-scratch", "epochs": EPOCHS, "lr": LR}
)

wandb.log({
    "final_val_map3": final_map3,
    "final_val_accuracy": final_acc,
    "final_val_f1": final_f1
})

wandb.finish()

## 9. Generate Submission

In [ ]:
top3 = np.argsort(-test_probs, axis=1)[:, :3]
preds = [
    (test_df.iloc[i]['id'], ' '.join(LABELS[j] for j in top3[i]))
    for i in range(len(test_df))
]

sub = pd.DataFrame(preds, columns=['ID', 'Prediction'])
sub = sub.sort_values('ID').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)

print('Submission saved!')
print(sub.head(10))